# Cell 1: Package Setup

In [1]:
import sys
!{sys.executable} -m pip install -q transformers sentencepiece nltk sacrebleu rouge-score
print("Done.")

Done.


# Cell 2: Paths & Configuration

In [2]:
import os, torch

JSON_PATH   = r'C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Local\stage2_english_reports.json'
WORKING_DIR = './BanglaCXR_Stage2'
os.makedirs(WORKING_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'JSON exists: {os.path.exists(JSON_PATH)}')

Device: cuda
JSON exists: True


# Cell 3: Imports

In [3]:
import json, re
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import nltk
nltk.download('punkt',      quiet=True)
nltk.download('punkt_tab',  quiet=True)
nltk.download('wordnet',    quiet=True)
print("Imports done.")

Imports done.


# Cell 4: Load JSON from Stage 1

In [4]:
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    stage1_data = json.load(f)

english_reports   = [d['english_report']   for d in stage1_data]
reference_reports = [d['reference_report'] for d in stage1_data]
filenames         = [d['filename']         for d in stage1_data]

print(f'Loaded {len(stage1_data)} reports from Stage 1 JSON.')
print(f'\nSample English report:')
print(english_reports[0][:200])

Loaded 550 reports from Stage 1 JSON.

Sample English report:
Findings: The lungs are clear. There is no focal airspace consolidation, pleural effusion or pneumothorax. Heart size and mediastinal contour within normal limits.
Impression: No acute cardiopulmonary


# Load NLLB-200 Translator

In [5]:
NLLB_MODEL_ID = 'facebook/nllb-200-distilled-600M'
print('Loading NLLB-200...')

nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_ID)
nllb_model     = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(DEVICE)
nllb_model.eval()

BANGLA_BOS_TOKEN_ID = nllb_tokenizer.convert_tokens_to_ids('ben_Beng')
print(f'NLLB-200 loaded. Bangla BOS token ID: {BANGLA_BOS_TOKEN_ID}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading NLLB-200...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

NLLB-200 loaded. Bangla BOS token ID: 256026
VRAM: 1.23 GB


In [ ]:
## Cell 6: Medical Glossary + Translation Function

In [6]:
MEDICAL_GLOSSARY = {
    'pneumothorax':     'নিউমোথোরাক্স',
    'pleural effusion': 'প্লুরাল ইফিউশন',
    'consolidation':    'কনসোলিডেশন',
    'cardiomegaly':     'কার্ডিওমেগালি',
    'atelectasis':      'অ্যাটেলেকটাসিস',
    'pneumonia':        'নিউমোনিয়া',
    'pulmonary edema':  'পালমোনারি এডিমা',
    'edema':            'এডিমা',
    'nodule':           'নডিউল',
    'fracture':         'ফ্র্যাকচার',
    'opacity':          'অপাসিটি',
    'infiltrate':       'ইনফিল্ট্রেট',
    'effusion':         'ইফিউশন',
    'mediastinum':      'মিডিয়াস্টিনাম',
    'cardiomediastinal':'কার্ডিওমিডিয়াস্টিনাল',
    'diaphragm':        'ডায়াফ্রাম',
    'hilum':            'হাইলাম',
    'trachea':          'ট্রাকিয়া',
    'thorax':           'থোরাক্স',
}

def translate_to_bangla(english_text, max_length=300):
    """Translate English radiology report to Bangla using NLLB-200."""
    if not english_text or not english_text.strip():
        return ''

    # Apply medical glossary BEFORE translation
    processed = english_text
    for eng, bn in MEDICAL_GLOSSARY.items():
        processed = re.sub(
            r'(?i)\b' + re.escape(eng) + r'\b',
            bn, processed
        )

    # Tokenize
    inputs = nllb_tokenizer(
        processed,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    ).to(DEVICE)

    with torch.no_grad():
        translated_tokens = nllb_model.generate(
            **inputs,
            forced_bos_token_id=BANGLA_BOS_TOKEN_ID,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    bangla = nllb_tokenizer.batch_decode(
        translated_tokens, skip_special_tokens=True
    )[0]
    return bangla.strip()


# Quick test
test_en = english_reports[0]
test_bn = translate_to_bangla(test_en)
print('=== TRANSLATION TEST ===')
print(f'EN: {test_en[:200]}')
print(f'BN: {test_bn[:200]}')

=== TRANSLATION TEST ===
EN: Findings: The lungs are clear. There is no focal airspace consolidation, pleural effusion or pneumothorax. Heart size and mediastinal contour within normal limits.
Impression: No acute cardiopulmonary
BN: ফলাফলঃ ফুসফুস পরিষ্কার। কোনও ফোকাল এয়ারস্পেস কনসোলাইডেশন, প্লুরাল ইফিউশন বা নিউমোথোরাক্স নেই। হৃদয়ের আকার এবং মধ্যস্থতাকারী কনট্যুর স্বাভাবিক সীমার মধ্যে। ইমপ্রেশনঃ কোনও তীব্র কার্ডিওপুলমোনার অস্বাভ


## Cell 7: Translate All 550 Reports

In [7]:
print(f'Translating {len(english_reports)} reports to Bangla...')
bangla_reports = []

for i, en_report in enumerate(english_reports):
    bn = translate_to_bangla(en_report)
    bangla_reports.append(bn)
    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/{len(english_reports)}] {bn[:80]}...')

print(f'\nDone. {len(bangla_reports)} Bangla reports generated.')

# Save combined JSON
output = []
for i, d in enumerate(stage1_data):
    output.append({
        'id':               d['id'],
        'filename':         d['filename'],
        'english_report':   d['english_report'],
        'bangla_report':    bangla_reports[i],
        'reference_report': d['reference_report']
    })

out_path = os.path.join(WORKING_DIR, 'stage2_bangla_reports.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f'✓ Saved → {out_path}')

# Show 3 samples
print('\n=== SAMPLE OUTPUTS ===')
for i in [0, 1, 2]:
    print(f'\n--- Sample {i+1} ---')
    print(f'EN: {output[i]["english_report"][:150]}')
    print(f'BN: {output[i]["bangla_report"][:150]}')

Translating 550 reports to Bangla...
  [50/550] ফলাফলঃ ফুসফুস পরিষ্কার। কোন ফোকাল এয়ারস্পেস কনসোলাইডেশন, প্লুরাল ইফিউশন বা নিউম...
  [100/550] ফলাফলঃ ফুসফুস পরিষ্কার। কোন প্লুরাল ইফিউশন বা নিউমোথোরাক্স নেই। ডান নীচের লব নডি...
  [150/550] ফলাফলঃ ফুসফুস পরিষ্কার। কোনও প্লুরাল ইফিউশন বা নিউমোথোরাক্স নেই। ডান ফুসফের বেসে...
  [200/550] ফলাফলঃ ইমপ্রেশনঃ হার্টের আকার স্বাভাবিক। ফুসফুস পরিষ্কার, ফোকাল এয়ারস্পেস রোগ ব...
  [250/550] ফলাফলঃ বাম নীচের লোবে একটি 3 সেন্টিমিটার নোডুলার অপাসিটি রয়েছে। ডান হিলার ক্যাল...
  [300/550] ফলাফলঃ ইমপ্রেশনঃ হৃৎপিণ্ডের আকার স্বাভাবিক। ফুসফুস পরিষ্কার এবং প্রসারিত, কোন ফো...
  [350/550] ফলাফলঃ ফুসফুস পরিষ্কার। কোন ফোকাল এয়ারস্পেস কনসোলাইডেশন, প্লুরাল ইফিউশন বা নিউম...
  [400/550] ফলাফলঃ ফুসফুস পরিষ্কার। কোন ফোকাল এয়ারস্পেস রোগ, ইফিউশন বা নিউমোথোরাক্স নেই। স্...
  [450/550] ফলাফলঃ ফুসফুস পরিষ্কার। কোন ফোকাল এয়ারস্পেস কনসোলাইডেশন, প্লুরাল ইফিউশন বা নিউম...
  [500/550] ফলাফলঃ স্বাভাবিক হৃদপিন্ডের আকার এবং মধ্যস্থতাকারী কন্ট্রোল। কোন ফোকাল এয়ারস্পে...
  [5

## Cell 8: Evaluation — BLEU & chrF++ on Bangla

In [8]:
import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

# ── English side metrics (vs reference) ───────────────────────
smoother  = SmoothingFunction().method1
refs_tok  = [[r.split()] for r in reference_reports]
hyps_tok  = [g.split()   for g in english_reports]

bleu1 = corpus_bleu(refs_tok, hyps_tok, weights=(1,0,0,0),             smoothing_function=smoother)
bleu2 = corpus_bleu(refs_tok, hyps_tok, weights=(0.5,0.5,0,0),         smoothing_function=smoother)
bleu4 = corpus_bleu(refs_tok, hyps_tok, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother)

rouge  = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
r1, r2, rL = [], [], []
for gen, ref in zip(english_reports, reference_reports):
    s = rouge.score(ref, gen)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)

meteor = np.mean([
    meteor_score([r.split()], g.split())
    for r, g in zip(reference_reports, english_reports)
])

# ── Bangla side: chrF++ (character-level, language-agnostic) ──
chrf_scores = [
    sacrebleu.corpus_chrf([bn], [[en]]).score
    for bn, en in zip(bangla_reports, english_reports)
]
chrf_en_bn = np.mean(chrf_scores)

# ── Summary ────────────────────────────────────────────────────
print('\n' + '='*58)
print('   BanglaCXR Stage 2 — Complete Evaluation')
print('='*58)
print(f'  Test samples         : {len(english_reports)}')
print(f'')
print(f'  English Report Quality (vs IU-Xray reference)')
print(f'  ── BLEU-1            : {bleu1:.4f}')
print(f'  ── BLEU-2            : {bleu2:.4f}')
print(f'  ── BLEU-4            : {bleu4:.4f}')
print(f'  ── ROUGE-1           : {np.mean(r1):.4f}')
print(f'  ── ROUGE-2           : {np.mean(r2):.4f}')
print(f'  ── ROUGE-L           : {np.mean(rL):.4f}')
print(f'  ── METEOR            : {meteor:.4f}')
print(f'')
print(f'  Bangla Translation Quality')
print(f'  ── chrF++ (EN→BN)    : {chrf_en_bn:.4f}')
print('='*58)


   BanglaCXR Stage 2 — Complete Evaluation
  Test samples         : 550

  English Report Quality (vs IU-Xray reference)
  ── BLEU-1            : 0.2884
  ── BLEU-2            : 0.1664
  ── BLEU-4            : 0.0699
  ── ROUGE-1           : 0.4352
  ── ROUGE-2           : 0.1604
  ── ROUGE-L           : 0.3103
  ── METEOR            : 0.2820

  Bangla Translation Quality
  ── chrF++ (EN→BN)    : 2.0786


## Cell 9: Create Bangla Reference & Proper Evaluation

In [10]:
print('Translating reference reports to Bangla')
bangla_references = []

for i, ref in enumerate(reference_reports):
    bn_ref = translate_to_bangla(ref)
    bangla_references.append(bn_ref)
    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/{len(reference_reports)}] done')

print(f'✓ {len(bangla_references)} Bangla reference reports created.')

# Save
for i, d in enumerate(output):
    d['bangla_reference'] = bangla_references[i]

out_path2 = os.path.join(WORKING_DIR, 'stage2_bangla_full.json')
with open(out_path2, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f'✓ Saved full data → {out_path2}')


# ── Proper Bangla Evaluation ───────────────────────────────────
import sacrebleu
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

smoother = SmoothingFunction().method1

# BLEU on Bangla (generated vs translated reference)
bn_refs_tok = [[r.split()] for r in bangla_references]
bn_hyps_tok = [g.split()   for g in bangla_reports]

bn_bleu1 = corpus_bleu(bn_refs_tok, bn_hyps_tok, weights=(1,0,0,0),             smoothing_function=smoother)
bn_bleu2 = corpus_bleu(bn_refs_tok, bn_hyps_tok, weights=(0.5,0.5,0,0),         smoothing_function=smoother)
bn_bleu4 = corpus_bleu(bn_refs_tok, bn_hyps_tok, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother)

# chrF++ on Bangla (character-level, robust for Bangla script)
bn_chrf = np.mean([
    sacrebleu.corpus_chrf([g], [[r]]).score
    for g, r in zip(bangla_reports, bangla_references)
])

# ── Final Complete Summary ─────────────────────────────────────
print('\n' + '='*60)
print('   BanglaCXR — Complete Two-Stage Evaluation')
print('='*60)
print(f'  Test samples              : {len(english_reports)}')
print(f'')
print(f'  Stage 1: English Report Generation (vs IU-Xray EN ref)')
print(f'  ── BLEU-1                 : {bleu1:.4f}')
print(f'  ── BLEU-2                 : {bleu2:.4f}')
print(f'  ── BLEU-4                 : {bleu4:.4f}')
print(f'  ── ROUGE-1                : {np.mean(r1):.4f}')
print(f'  ── ROUGE-2                : {np.mean(r2):.4f}')
print(f'  ── ROUGE-L                : {np.mean(rL):.4f}')
print(f'  ── METEOR                 : {meteor:.4f}')
print(f'')
print(f'  Stage 2: Bangla Translation (vs NLLB Bangla reference)')
print(f'  ── BLEU-1 (BN)            : {bn_bleu1:.4f}')
print(f'  ── BLEU-2 (BN)            : {bn_bleu2:.4f}')
print(f'  ── BLEU-4 (BN)            : {bn_bleu4:.4f}')
print(f'  ── chrF++ (BN)            : {bn_chrf:.4f}')
print('='*60)

# Sample outputs
print('\n=== SAMPLE BANGLA REPORTS ===')
for i in range(3):
    print(f'\n--- Sample {i+1} ---')
    print(f'EN GEN : {output[i]["english_report"][:120]}')
    print(f'BN GEN : {output[i]["bangla_report"][:120]}')
    print(f'BN REF : {bangla_references[i][:120]}')

Translating reference reports to Bangla
  [50/550] done
  [100/550] done
  [150/550] done
  [200/550] done
  [250/550] done
  [300/550] done
  [350/550] done
  [400/550] done
  [450/550] done
  [500/550] done
  [550/550] done
✓ 550 Bangla reference reports created.
✓ Saved full data → ./BanglaCXR_Stage2\stage2_bangla_full.json

   BanglaCXR — Complete Two-Stage Evaluation
  Test samples              : 550

  Stage 1: English Report Generation (vs IU-Xray EN ref)
  ── BLEU-1                 : 0.2884
  ── BLEU-2                 : 0.1664
  ── BLEU-4                 : 0.0699
  ── ROUGE-1                : 0.4352
  ── ROUGE-2                : 0.1604
  ── ROUGE-L                : 0.3103
  ── METEOR                 : 0.2820

  Stage 2: Bangla Translation (vs NLLB Bangla reference)
  ── BLEU-1 (BN)            : 0.2626
  ── BLEU-2 (BN)            : 0.1358
  ── BLEU-4 (BN)            : 0.0502
  ── chrF++ (BN)            : 35.1756

=== SAMPLE BANGLA REPORTS ===

--- Sample 1 ---
EN GEN : Findings: